In [ ]:
import glob
import argparse
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import oggm
from oggm import utils

%matplotlib inline

In [ ]:
# Download filen
utils.get_rgi_dir(version='62')

In [ ]:
path_O1_shp = 'C:\\Users\\emma\\OGGM\\rgi\\RGIV62/00_rgi62_regions/00_rgi62_O1Regions.shp' # shp file of regional boundaries

# Import 20-bin gridded training dataset
metadata_file = "C:\\Users\\emma\\Desktop\\AML\\metadata19_hmineq0.0_tmin20050000_mean_grid_20.csv"
glathida_rgis = pd.read_csv(metadata_file, low_memory=False)

In [ ]:
# See all RGI's regional boundaries
# See pag 8 at https://nsidc.org/sites/nsidc.org/files/technical-references/RGI_Tech_Report_V6.0.pdf
world = gpd.read_file(path_O1_shp)      # import as geopandas dataframe
print(world)
fig, ax = plt.subplots()
for n, rgi_series in enumerate(world['geometry']):
    x, y = rgi_series.exterior.xy
    min_x, max_x = min(x), max(x)
    min_y, max_y = min(y), max(y)
    ax.plot(x, y, c='g')
    ax.text(np.mean([min_x, max_x]), np.mean([min_y, max_y]), f"{world['RGI_CODE'].iloc[n]}", color='green', fontsize=9)
plt.show()

In [ ]:
# Let's select one region of interest, e.g. rgi=11 (Central Europe)

# Dataframe of glaciers for rgi=11. This dataset contains a lot of info, some of them are also used to create the
# training dataset. RGIId is the official name of glaciers according to the Randolph Glacier Inventory.
# In the column geometry you get the geometries.

rgi = 19
oggm_rgi_shp = glob.glob(f"C:/Users/emma/OGGM/rgi/RGIV62/{rgi}*/{rgi}*.shp")[0] # .shp file for rgi 11
oggm_rgi_glaciers = gpd.read_file(oggm_rgi_shp)                         # dataframe for rgi 11

In [ ]:
# Plot one specific glacier by passing the RGIId code.
# Example: RGI60-11.01450 is the Aletsch Glacier, the biggest in Central Europe (rgi=11).
# You under the geometry column you find the glacier external boundary (.external)
# and the glacier nunataks, if any (.interiors). Nunataks are the inner glacier portions that are deglaciated (=rock).
# We can also find the points in the training dataset for this glacier (not all glaciers contain measurements)

glacier_geometry = oggm_rgi_glaciers.loc[oggm_rgi_glaciers['RGIId']=='RGI60-11.01450']['geometry'].item()

# Get the measurements in the training dataset by passing the glacier id
glathida_rgis_aletsch = glathida_rgis.loc[glathida_rgis['RGIId']=='RGI60-11.01450']
print(f"We have {len(glathida_rgis_aletsch)} points in the training dataset")

fig, ax = plt.subplots()

exterior_ring = glacier_geometry.exterior # External geometry
ax.plot(*exterior_ring.xy, c='b')
ax.fill(*exterior_ring.xy, c='lightblue')
glacier_nunataks_list = [nunatak for nunatak in glacier_geometry.interiors] # Nunataks (list may be empty if no nunataks)
for nunatak in glacier_nunataks_list:
    ax.plot(*nunatak.xy, c='k', lw=0.8)
    ax.fill(*nunatak.xy, c='red')
ax.scatter(x=glathida_rgis_aletsch['POINT_LON'], y=glathida_rgis_aletsch['POINT_LAT'], s=20, c='orange', label=None)

# Get axis limits
xlim = ax.get_xlim()
ylim = ax.get_ylim()

    # Calculate scale
glacier_scale_x = xlim[1] - xlim[0]
glacier_scale_y = ylim[1] - ylim[0]

ax.axis('off')

plt.show()

print("Glacier scale in x-direction:", glacier_scale_x)
print("Glacier scale in y-direction:", glacier_scale_y)

The code below is used for generating images for the autoencoder.

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

# Define the path to the desired directory in Colab environment
output_dir = "C:/Users/emma/Desktop/AML/GlacierPlots3"

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Set the desired width in inches (width_in_inches = width_in_pixels / dpi)
desired_width_pixels = 64
dpi = 64
width_in_inches = desired_width_pixels / dpi
height_in_inches = width_in_inches  # Keep the aspect ratio square for simplicity

# Function to plot and save a specific glacier
def plot_and_save_glacier(glacier_geometry, glathida_rgis, rgi_id, output_dir, dpi=64):
    glathida_rgis_aletsch = glathida_rgis.loc[glathida_rgis['RGIId'] == rgi_id]

    fig, ax = plt.subplots(figsize=(width_in_inches, height_in_inches), dpi=dpi)
    exterior_ring = glacier_geometry.exterior
    ax.plot(*exterior_ring.xy, c='b')
    ax.fill(*exterior_ring.xy, c='lightblue')
    glacier_nunataks_list = [nunatak for nunatak in glacier_geometry.interiors]
    for nunatak in glacier_nunataks_list:
        ax.plot(*nunatak.xy, c='k', lw=0.8)
        ax.fill(*nunatak.xy, c='red')
    ax.scatter(x=glathida_rgis_aletsch['POINT_LON'], y=glathida_rgis_aletsch['POINT_LAT'], s=20, c='orange', label=None)

    # Get axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # Calculate scale
    glacier_scale_x = xlim[1] - xlim[0]
    glacier_scale_y = ylim[1] - ylim[0]

    ax.axis('off')

    # Save the plot as a PNG file in the specified directory
    output_path = os.path.join(output_dir, f'{rgi_id}.png')
    plt.savefig(output_path, dpi=dpi)
    plt.close(fig)  # Close the figure to save memory

    return glacier_scale_x, glacier_scale_y

# Assuming oggm_rgi_glaciers is your DataFrame
unique_rgi_ids = oggm_rgi_glaciers['RGIId'].unique()

# Convert to list for easier iteration
unique_rgi_ids_list = unique_rgi_ids.tolist()


# Create an empty list to store the rows
glacier_scale_data = pd.DataFrame(columns=['glacierID', 'x_scale', 'y_scale','Form'])
rows = []

# Iterate over the first 100 unique RGIIds and save plots
for rgi_id in unique_rgi_ids_list:
    glacier_geometry = oggm_rgi_glaciers.loc[oggm_rgi_glaciers['RGIId'] == rgi_id]['geometry'].item()
    x_scale, y_scale = plot_and_save_glacier(glacier_geometry, glathida_rgis, rgi_id, output_dir)
    # Append row to the list
    rows.append({'glacierID': rgi_id, 'x_scale': x_scale, 'y_scale': y_scale, 'Form': oggm_rgi_glaciers.loc[oggm_rgi_glaciers['RGIId'] == rgi_id]['Form']})

# Concatenate the list of rows with glacier_scale_data
glacier_scale_data = pd.concat([glacier_scale_data, pd.DataFrame(rows)])

# Print the DataFrame
print(glacier_scale_data)


In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

# Define the path to the desired directory in Colab environment
output_dir = "C:/Users/emma/Desktop/AML/GlacierPlots3"

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Set the desired width in inches (width_in_inches = width_in_pixels / dpi)
desired_width_pixels = 64
dpi = 64
width_in_inches = desired_width_pixels / dpi
height_in_inches = width_in_inches  # Keep the aspect ratio square for simplicity

# Function to plot and save a specific glacier
def plot_and_save_glacier(glacier_geometry, glathida_rgis, rgi_id, output_dir, dpi=64):
    glathida_rgis_aletsch = glathida_rgis.loc[glathida_rgis['RGIId'] == rgi_id]

    glacier_nunataks_list = [nunatak for nunatak in glacier_geometry.interiors]
    nunataks_bool = False
    for nunatak in glacier_nunataks_list:
        nunataks_bool = True


    
    return nunataks_bool

# Assuming oggm_rgi_glaciers is your DataFrame
unique_rgi_ids = oggm_rgi_glaciers['RGIId'].unique()

# Convert to list for easier iteration
unique_rgi_ids_list = unique_rgi_ids.tolist()


# Create an empty list to store the rows
glacier_scale_data = pd.DataFrame(columns=['glacierID','Nunataks'])
rows = []

# Iterate over the first 100 unique RGIIds and save plots
for rgi_id in unique_rgi_ids_list:
    glacier_geometry = oggm_rgi_glaciers.loc[oggm_rgi_glaciers['RGIId'] == rgi_id]['geometry'].item()
    nunataks_boolean = plot_and_save_glacier(glacier_geometry, glathida_rgis, rgi_id, output_dir)
    # Append row to the list
    rows.append({'glacierID': rgi_id, 'Nunataks': nunataks_boolean})

# Concatenate the list of rows with glacier_scale_data
glacier_scale_data = pd.concat([glacier_scale_data, pd.DataFrame(rows)])

# Print the DataFrame
print(glacier_scale_data)


In [ ]:
#len(oggm_rgi_glaciers['RGIId'].unique().tolist())
sum(glacier_scale_data['Nunataks'])

In [ ]:

# Define the path to save the CSV file
#csv_file_path = 'C:/Users/emma/Desktop/AML/glacier_data_RGI19.csv'
csv_file_path = 'C:/Users/emma/Desktop/AML/glacier_data_RGI19_nunatak.csv'

# Save glacier_data as a CSV file
glacier_scale_data.to_csv(csv_file_path, index=False)


In [ ]:
# Plot the whole regional set of glaciers and all nunataks
# zoom in with matplotlib in interactive mode (%matplotlib qt ) to have a sense of all glacier and their nunataks

region11 = world.loc[12] # Regional boundary
fig, ax = plt.subplots()
for glacier_rgiid, glacier_geometry in zip(oggm_rgi_glaciers['RGIId'], oggm_rgi_glaciers['geometry']):

    exterior_ring = glacier_geometry.exterior
    glacier_nunataks_list = [nunatak for nunatak in glacier_geometry.interiors]

    ax.plot(*exterior_ring.xy, c='b')
    for nunatak in glacier_nunataks_list:
        ax.plot(*nunatak.xy, c='k', lw=0.8) # Plot glacier nunataks (if any)
ax.plot(*region11['geometry'].exterior.xy, c='g') # Regional boundary
ax.text(min(*region11['geometry'].exterior.xy[0])+1,
        max(*region11['geometry'].exterior.xy[1])-1, f'rgi {rgi}', c='g', fontsize=12)
plt.show()

In [ ]:
oggm_rgi_glaciers.describe()

In [ ]:
oggm_rgi_glaciers.shape


---------------- Prøver at bygge en autoencoder -------------------


---- Prøver med test og træning, med spejlvendte og billeder på hovedet ----

nu med reduceret latent space

In [ ]:
# Angiv TensorFlow til at bruge den første GPU, hvis tilgængelig
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        tf.config.set_visible_devices(gpus[0], 'GPU')
        print("GPU found:", gpus[0])
    except RuntimeError as e:
        print(e)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models, losses, optimizers
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Flatten, Dense, Reshape, Cropping2D, Conv2DTranspose
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Funktion til at indlæse billeder fra en mappe
def load_images_from_directory(directory, target_size=(64, 64)):
    images = []
    filenames = []
    for filename in os.listdir(directory):
        if filename.endswith('.png'):
            img_path = os.path.join(directory, filename)
            img = load_img(img_path, target_size=target_size)
            img_array = img_to_array(img)
            images.append(img_array)
            filenames.append(filename)
    return np.array(images), filenames

# Funktion til at augmentere billeder
def augment_images(images):
    augmented_images = []
    for img in images:
        # Original
        augmented_images.append(img)
        # Spejlvendt horisontalt
        augmented_images.append(np.fliplr(img))
        # Spejlvendt vertikalt
        #augmented_images.append(np.flipud(img))
        # Spejlvendt horisontalt og vertikalt
        #augmented_images.append(np.flipud(np.fliplr(img)))
    return np.array(augmented_images)

# Sti til din billedmappe
image_directory = 'C:/Users/emma/Desktop/AML/GlacierPlots'
images, gletcher_ids1 = load_images_from_directory(image_directory, target_size=(64, 64))
image_directory2 = 'C:/Users/emma/Desktop/AML/GlacierPlots3'
images2, gletcher_ids2 = load_images_from_directory(image_directory2, target_size=(64, 64))

gletcher_ids = gletcher_ids1 + gletcher_ids2
images = np.concatenate((images, images2), axis=0)
del images2
del gletcher_ids1
del gletcher_ids2


# Normaliser pixelværdierne til intervallet [0, 1]
images = images.astype('float32') / 255.0

# Udvid datasættet
#images = augment_images(images)

# Split the data into training and validation sets
#images_to_use, discard, ids_to_use, discard2 = train_test_split(images,gletcher_ids,test_size = 0.5,random_state=42)
#X_train, X_val, ids_train, ids_val = train_test_split(images, gletcher_ids, test_size=0.2, random_state=42)
#print(len(images))
#del images
#del discard
#del gletcher_ids
#del discard2
#print(len(images_to_use))

#X_train, X_val, ids_train, ids_val = train_test_split(images_to_use, ids_to_use, test_size=0.2, random_state=42)
X_train, X_val, ids_train, ids_val = train_test_split(images, gletcher_ids, test_size=0.2, random_state=42)
#del images_to_use
#del ids_to_use
del images
del gletcher_ids

In [ ]:

latent_space_dim = 64
# Autoencoder model
input_img = Input(shape=(64, 64, 3))

# Encoder
x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Flatten()(x)

# Latent space with multiple dense layers
latent = Dense(256, activation='relu')(x)
latent = Dense(128, activation='relu')(latent)
latent = Dense(latent_space_dim, activation='relu')(latent)

# Decoder
x = Dense(128, activation='relu')(latent)
x = Dense(256, activation='relu')(x)
x = Dense(4 * 4 * 256, activation='relu')(x)
x = Reshape((4, 4, 256))(x)
x = UpSampling2D((2, 2))(x)
x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
decoded = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

    
autoencoder = Model(input_img, decoded)
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

# Train autoencoder
early_stopping = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-4)

history = autoencoder.fit(X_train, X_train, epochs=30, batch_size=32, validation_data=(X_val, X_val),callbacks=[early_stopping, reduce_lr])

# Visualize reconstructed images
decoded_imgs = autoencoder.predict(X_val)

In [ ]:

# Uddrag latent space
with tf.device('/CPU:0'):
    encoder = Model(input_img, latent)
    X_train_encoded = encoder.predict(X_train)
    X_val_encoded = encoder.predict(X_val)
    

In [ ]:
# Konverter til DataFrame (training problem)
train_df = pd.DataFrame(X_train_encoded, columns=[f'dim_{i}' for i in range(latent_space_dim)])
val_df = pd.DataFrame(X_val_encoded, columns=[f'dim_{i}' for i in range(latent_space_dim)])

def extract_glacier_id(filenames):
    return [os.path.splitext(filename)[0] for filename in filenames]

train_df['glacier_id'] = extract_glacier_id(ids_train)
val_df['glacier_id'] = extract_glacier_id(ids_val)

In [ ]:
val_df.head()

In [ ]:
# Find søjler med mindst én ikke-nul værdi
non_zero_columns = val_df.loc[:, (val_df != 0).any(axis=0)]

# Opret en ny DataFrame med kun de relevante søjler
val_df = val_df[non_zero_columns.columns]
train_df = train_df[non_zero_columns.columns]

In [ ]:
val_df.shape

In [ ]:
# Gem til csv-filer
#train_df.to_csv('train_latent_space.csv', index=False)
#val_df.to_csv('val_latent_space.csv', index=False)

pd.concat([train_df, val_df], axis=0).to_csv('latent_space_64_AE.csv',index=False)

In [ ]:
# Plot original og rekonstrueret billeder
index_to_start_from = 10

n = 10  # Antal billeder at plotte
plt.figure(figsize=(20, 4))
for i in range(n):
    # Original billede
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(X_val[index_to_start_from + i])
    plt.title("Original")
    plt.axis("off")

    # Rekonstrueret billede
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(autoencoder.predict(X_val[index_to_start_from + i].reshape(1, 64, 64, 3)).reshape(64, 64, 3))
    plt.title("Reconstructed")
    plt.axis("off")
plt.show()

In [ ]:
nunatak_df = pd.concat([pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI10_nunatak.csv"),
           pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI11_nunatak.csv"),
           pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI12_nunatak.csv"),
           pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI13_nunatak.csv"),
           pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI14_nunatak.csv"),
           pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI15_nunatak.csv")], axis=0)

nunatak_df = nunatak_df.rename(columns={'glacierID': 'glacier_id'})
# Perform the merge
train_df = pd.merge(train_df, nunatak_df, on="glacier_id", how="inner")
train_df = pd.merge(val_df, nunatak_df, on="glacier_id", how="inner")

In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K

# Clear the Keras session to free up memory
K.clear_session()
tf.compat.v1.reset_default_graph()
del autoencoder
del history

In [ ]:

import umap
import umap.plot
# Uddrag kun de latente dimensioner fra dine dataframes
target = train_df['Nunataks']
latent_space_data = train_df.drop(columns=['glacier_id','Nunataks']).values
mapper = umap.UMAP().fit(latent_space_data)
umap.plot.points(mapper,labels=target,theme = "fire")

In [ ]:

tf.keras.backend.clear_session()
gc.collect()
del vae


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, losses, optimizers, callbacks
import numpy as np
from tensorflow.keras.layers import LeakyReLU
import gc

# Define input dimensions
input_shape = (64, 64, 3)  # Change to (64, 64, 1) if grayscale

# Encoder
def build_encoder(input_shape, latent_dim):
    encoder_inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(encoder_inputs)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)
    x = layers.Flatten()(x)

    z_mean = layers.Dense(latent_dim, name='z_mean')(x)
    z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)

    def sampling(args):
        z_mean, z_log_var = args
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    z = layers.Lambda(sampling, output_shape=(latent_dim,), name='z')([z_mean, z_log_var])

    encoder = models.Model(encoder_inputs, [z_mean, z_log_var, z], name='encoder')
    return encoder

# Decoder
def build_decoder(latent_dim):
    latent_inputs = layers.Input(shape=(latent_dim,))

    x = layers.Dense(16 * 16 * 32, activation='relu')(latent_inputs)
    x = layers.Reshape((16, 16, 32))(x)
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(x)
    x = layers.UpSampling2D((2, 2))(x)
    decoder_outputs = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)  # 3 for color, change to 1 for grayscale

    decoder = models.Model(latent_inputs, decoder_outputs, name='decoder')
    return decoder

# Define the Variational Autoencoder (VAE) model
class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstructed = self.decoder(z)
        kl_loss = -0.5 * tf.reduce_mean(
            z_log_var - tf.square(z_mean) - tf.exp(z_log_var) + 1)
        kl_loss *= 0.01 #Sæt vægten her
        self.add_loss(kl_loss)
        return reconstructed

# Parameters
latent_dim = 32  # Adjust as needed

# Build encoder and decoder
encoder = build_encoder(input_shape, latent_dim)
decoder = build_decoder(latent_dim)

# Build VAE
vae = VAE(encoder, decoder)
optimizer = optimizers.Adam(learning_rate=0.001)
#vae.compile(optimizer=optimizer, loss=losses.MeanSquaredError())
vae.compile(optimizer=optimizer, loss=losses.BinaryCrossentropy())
 
# Print model summaries
encoder.summary()
decoder.summary()

# Ensure data shapes are correct before training
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_val: {X_val.shape}")

early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-4)

vae.fit(X_train, X_train, epochs=30, batch_size=32, validation_data=(X_val, X_val), callbacks=[early_stopping, reduce_lr])
vae.summary()


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, losses, optimizers, callbacks
import numpy as np
from tensorflow.keras.layers import LeakyReLU
import gc

# Define input dimensions
input_shape = (64, 64, 3)  # Change to (64, 64, 1) if grayscale

# Encoder
def build_encoder(input_shape, latent_dim):
    encoder_inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(encoder_inputs)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)
    x = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)
    x = layers.Flatten()(x)

    z_mean = layers.Dense(latent_dim, name='z_mean')(x)
    z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)

    def sampling(args):
        z_mean, z_log_var = args
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    z = layers.Lambda(sampling, output_shape=(latent_dim,), name='z')([z_mean, z_log_var])

    encoder = models.Model(encoder_inputs, [z_mean, z_log_var, z], name='encoder')
    return encoder

# Decoder
def build_decoder(latent_dim):
    latent_inputs = layers.Input(shape=(latent_dim,))

    x = layers.Dense(4 * 4 * 256, activation='relu')(latent_inputs)
    x = layers.Reshape((4, 4, 256))(x)
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.UpSampling2D((2, 2))(x)
    decoder_outputs = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)  # 3 for color, change to 1 for grayscale

    decoder = models.Model(latent_inputs, decoder_outputs, name='decoder')
    return decoder

# Define the Variational Autoencoder (VAE) model
class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstructed = self.decoder(z)
        kl_loss = -0.5 * tf.reduce_mean(
            z_log_var - tf.square(z_mean) - tf.exp(z_log_var) + 1)
        kl_loss *= 0.01 #Sæt vægten her
        self.add_loss(kl_loss)
        return reconstructed

# Parameters
latent_dim = 64  # Adjust as needed

# Build encoder and decoder
encoder = build_encoder(input_shape, latent_dim)
decoder = build_decoder(latent_dim)

# Build VAE
vae = VAE(encoder, decoder)
optimizer = optimizers.Adam(learning_rate=0.001)
#vae.compile(optimizer=optimizer, loss=losses.MeanSquaredError())
vae.compile(optimizer=optimizer, loss=losses.BinaryCrossentropy())
 
# Print model summaries
encoder.summary()
decoder.summary()

# Ensure data shapes are correct before training
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_val: {X_val.shape}")

early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-4)

vae.fit(X_train, X_train, epochs=30, batch_size=32, validation_data=(X_val, X_val), callbacks=[early_stopping, reduce_lr])
vae.summary()


In [ ]:
#Til VAE
# Uddrag latent space predictions
with tf.device('/CPU:0'):
    X_train_encoded, _, _ = vae.encoder.predict(X_train)
    X_val_encoded, _, _ = vae.encoder.predict(X_val)
    


In [ ]:
latent_space_dim = 32
# Konverter til DataFrame (training problem)
train_df = pd.DataFrame(X_train_encoded, columns=[f'dim_{i}' for i in range(latent_space_dim)])
val_df = pd.DataFrame(X_val_encoded, columns=[f'dim_{i}' for i in range(latent_space_dim)])

def extract_glacier_id(filenames):
    return [os.path.splitext(filename)[0] for filename in filenames]

train_df['glacier_id'] = extract_glacier_id(ids_train)
val_df['glacier_id'] = extract_glacier_id(ids_val)

In [ ]:
val_df.head()

In [ ]:
# Find søjler med mindst én ikke-nul værdi
non_zero_columns = val_df.loc[:, (val_df != 0).any(axis=0)]

# Opret en ny DataFrame med kun de relevante søjler
val_df = val_df[non_zero_columns.columns]
train_df = train_df[non_zero_columns.columns]

In [ ]:
# Gem til csv-filer
#train_df.to_csv('train_latent_space.csv', index=False)
#val_df.to_csv('val_latent_space.csv', index=False)

#pd.concat([train_df, val_df], axis=0).to_csv('latent_space_32.csv',index=False)


In [ ]:
# Plot original og rekonstrueret billeder
index_to_start_from = 30

n = 10  # Antal billeder at plotte
plt.figure(figsize=(20, 4))
for i in range(n):
    # Original billede
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(X_val[index_to_start_from + i])
    plt.title("Original")
    plt.axis("off")

    # Rekonstrueret billede fra VAE
    ax = plt.subplot(2, n, i + 1 + n)
    original_image = X_val[index_to_start_from + i]
    original_image = np.expand_dims(original_image, axis=0)  # Tilføj en ekstra dimension for batch
    reconstructed_image = vae.predict(original_image)
    reconstructed_image = reconstructed_image.reshape(64, 64, 3)
    plt.imshow(reconstructed_image)
    plt.title("Reconstructed")
    plt.axis("off")
plt.show()

In [ ]:
nunatak_df = pd.concat([pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI10_nunatak.csv"),
           pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI11_nunatak.csv"),
           pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI12_nunatak.csv"),
           pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI13_nunatak.csv"),
           pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI16_nunatak.csv"),
           pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI17_nunatak.csv"),
           pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI18_nunatak.csv"),
           pd.read_csv("C:/Users/emma/Desktop/AML/glacier_data_RGI19_nunatak.csv")], axis=0)

nunatak_df = nunatak_df.rename(columns={'glacierID': 'glacier_id'})
# Perform the merge
train_df = pd.merge(train_df, nunatak_df, on="glacier_id", how="inner")
val_df = pd.merge(val_df, nunatak_df, on="glacier_id", how="inner")

In [ ]:
import umap
import umap.plot
# Uddrag kun de latente dimensioner fra dine dataframes
target = train_df['Nunataks']
latent_space_data = train_df.drop(columns=['glacier_id','Nunataks']).values
mapper = umap.UMAP().fit(latent_space_data)
umap.plot.points(mapper,labels=target,theme = "fire")

In [ ]:
# Ekstrakter embeddings
embedding = mapper.embedding_

# Beregn centret af embeddings
center = np.mean(embedding, axis=0)

# Beregn afstand fra centret for hver punkt
distances = np.linalg.norm(embedding - center, axis=1)

# Bestem en grænse for at filtrere punkter (her bruger vi 95 percentilen af afstandene)
threshold = np.percentile(distances, 99)

# Filtrer punkter, der er inden for threshold afstanden
filtered_indices = distances < threshold
filtered_embedding = embedding[filtered_indices]
filtered_labels = train_df['Nunataks'].values[filtered_indices]

# Set up farver og plotparametre
colors = np.where(filtered_labels == 1, 'red', '#00009B')  # Farve rød for nunataks og blå for ingen nunataks
alpha = 0.5  # Gennemsigtighed
point_size = 1.5  # Punktstørrelse

# Plot de filtrerede embeddings
plt.figure(figsize=(10, 8))
plt.scatter(filtered_embedding[:, 0], filtered_embedding[:, 1], c=colors, s=point_size, alpha=alpha)

# Tilpas plot udseende
plt.title("Filtered UMAP Embeddings")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.grid(False)  # Ingen grid
plt.gca().set_facecolor('black')  # Sort baggrund
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, losses, optimizers, callbacks
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
import keras.backend as K
import gc

# Define input dimensions
input_shape = (64, 64, 3)  # Change to (64, 64, 1) if grayscale

def sampling(args):
    z_mean, z_log_var = args
    batch = tf.shape(z_mean)[0]
    dim = tf.shape(z_mean)[1]
    epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

def create_vae(params):
    # Encoder
    encoder_inputs = layers.Input(shape=input_shape)
    x = encoder_inputs
    filters = params['conv_filters']
    for _ in range(int(params['conv_layers'])):
        x = layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
        x = layers.MaxPooling2D((2, 2), padding='same')(x)
        filters *= 2

    x = layers.Flatten()(x)
    
    units = params['dense_units']
    for _ in range(int(params['dense_layers'])):
        x = layers.Dense(units, activation='relu')(x)
        units //= 2

    z_mean = layers.Dense(int(params['latent_dim']), name='z_mean')(x)
    z_log_var = layers.Dense(int(params['latent_dim']), name='z_log_var')(x)
    z = layers.Lambda(sampling, output_shape=(int(params['latent_dim']),), name='z')([z_mean, z_log_var])

    encoder = models.Model(encoder_inputs, [z_mean, z_log_var, z], name='encoder')
    
    # Decoder
    latent_inputs = layers.Input(shape=(int(params['latent_dim']),))
    units = int(params['latent_dim']) * 2
    x = layers.Dense(units, activation='relu')(latent_inputs)
    for _ in range(int(params['dense_layers'])):
        x = layers.Dense(units, activation='relu')(x)
        units *= 2

    decoder_input_shape = 64 // (2 ** int(params['conv_layers']))
    x = layers.Dense(decoder_input_shape * decoder_input_shape * filters // 2, activation='relu')(x)
    x = layers.Reshape((decoder_input_shape, decoder_input_shape, filters // 2))(x)
    for _ in range(int(params['conv_layers'])):
        x = layers.Conv2D(filters // 2, (3, 3), activation='relu', padding='same')(x)
        x = layers.UpSampling2D((2, 2))(x)
        filters //= 2
    
    decoder_outputs = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)
    decoder = models.Model(latent_inputs, decoder_outputs, name='decoder')

    # Define the VAE
    class VAE(models.Model):
        def __init__(self, encoder, decoder, **kwargs):
            super(VAE, self).__init__(**kwargs)
            self.encoder = encoder
            self.decoder = decoder

        def call(self, inputs):
            z_mean, z_log_var, z = self.encoder(inputs)
            reconstructed = self.decoder(z)
            kl_loss = -0.5 * tf.reduce_mean(
                z_log_var - tf.square(z_mean) - tf.exp(z_log_var) + 1)
            kl_loss *= 0.01  # KL weight
            self.add_loss(kl_loss)
            return reconstructed

    vae = VAE(encoder, decoder)
    vae.compile(optimizer=optimizers.Adam(learning_rate=0.001),
                loss=losses.BinaryCrossentropy())

    return vae

def objective(params):
    # Clear previous session to free up memory
    K.clear_session()
    tf.keras.backend.clear_session()
    gc.collect()
    
    vae = create_vae(params)
    history = vae.fit(X_train, X_train, epochs=12, batch_size=32, validation_data=(X_val, X_val), verbose=0)
    val_loss = np.min(history.history['val_loss'])
    
    # Clear session and delete the model and history
    K.clear_session()
    tf.keras.backend.clear_session()
    del vae
    del history
    gc.collect()
    
    return {'loss': val_loss, 'status': STATUS_OK}
    return {'loss': val_loss, 'status': STATUS_OK}

# Define the hyperparameter space
param_space = {
    'conv_layers': hp.choice('conv_layers', [2, 3, 4]),
    'conv_filters': hp.choice('conv_filters', [16, 32, 64, 128]),
    'dense_layers': hp.choice('dense_layers', [0, 1, 2, 3]),
    'dense_units': hp.choice('dense_units', [128, 256, 512]),
    'latent_dim': hp.choice('latent_dim', [24])
}



# Use Hyperopt to find the best hyperparameters
trials = Trials()
best_params = fmin(objective, param_space, algo=tpe.suggest, max_evals=30, trials=trials)

print("Best hyperparameters: ", best_params)


In [ ]:
# Extract the trials information
trials_dict = {
    'trial_number': [],
    'parameters': [],
    'loss': [],
    'status': []
}

for trial in trials.trials:
    trials_dict['trial_number'].append(trial['tid'])
    trials_dict['parameters'].append(trial['misc']['vals'])
    trials_dict['loss'].append(trial['result']['loss'])
    trials_dict['status'].append(trial['result']['status'])

# Convert to DataFrame for easier viewing
trials_df = pd.DataFrame(trials_dict)

trials_df = trials_df.sort_values(by='loss', ascending=True)


In [ ]:
trials_df.head()

In [ ]:

trials_df.iloc[2,1]

In [ ]:
train_df.shape

In [ ]:
pd.read_csv("C:/Users/emma/Desktop/AML/latent_space_32.csv").shape

In [ ]:
pd.read_csv("C:/Users/emma/Desktop/AML/latent_space_48.csv")['glacier_id'].isna().sum()